# Development of CNNs Models with Log-Mel Spectrogram and MFCC as their input features:

## Imports:

In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import random

In [3]:
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## Geral:

In [4]:
SAMPLE_RATE = 16000
N_FFT = 512
HOP_LENGTH = 160
N_MELS = 64
BATCH_SIZE = 64

CLASS_NAMES = np.array(['go', 'no', 'off', 'on', 'stop', '_unknown_'])
NUM_CLASSES = len(CLASS_NAMES)

AUTOTUNE = tf.data.AUTOTUNE

DATASET_PATH = "../Datasets/KWS/KWS/dataset_augmented/"

In [5]:
def get_label(file_path):
    parts = tf.strings.split(file_path, os.path.sep)
    label = parts[-2]
    return tf.argmax(label == CLASS_NAMES)

In [6]:
def process_audio_log_mel(file_path):
    audio_binary = tf.io.read_file(file_path)
    audio, _ = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)

    audio = audio[:SAMPLE_RATE]
    audio = tf.pad(audio, [[0, SAMPLE_RATE - tf.shape(audio)[0]]])

    stft = tf.signal.stft(audio, frame_length=N_FFT, frame_step=HOP_LENGTH, fft_length=N_FFT)
    spectrogram = tf.abs(stft)

    num_spectrogram_bins = spectrogram.shape[-1]

    linear_to_mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MELS,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz= 20,
        upper_edge_hertz= SAMPLE_RATE / 2
    )

    mel_spectrogram = tf.tensordot(spectrogram, linear_to_mel_weight_matrix, 1)
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

    means = tf.math.reduce_mean(log_mel_spectrogram)
    stds = tf.math.reduce_std(log_mel_spectrogram)

    log_mel_spectrogram = (log_mel_spectrogram - means) / stds

    return tf.expand_dims(log_mel_spectrogram, -1)

In [7]:
def load_data_log_mel(file_path):
    label = get_label(file_path)
    spectrogram = process_audio_log_mel(file_path)
    return spectrogram, label

In [8]:
def process_audio_MFCC(file_path):
    audio_binary = tf.io.read_file(file_path)
    audio, _ = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)

    audio = audio[:SAMPLE_RATE]
    audio = tf.pad(audio, [[0, SAMPLE_RATE - tf.shape(audio)[0]]])

    stft = tf.signal.stft(audio, frame_length=N_FFT, frame_step=HOP_LENGTH, fft_length=N_FFT)
    spectrogram = tf.abs(stft)

    num_spectrogram_bins = spectrogram.shape[-1]

    linear_to_mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MELS,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz= 20,
        upper_edge_hertz= SAMPLE_RATE / 2
    )

    mel_spectrogram = tf.tensordot(spectrogram, linear_to_mel_weight_matrix, 1)
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

    means = tf.math.reduce_mean(log_mel_spectrogram)
    stds = tf.math.reduce_std(log_mel_spectrogram)

    log_mel_spectrogram = (log_mel_spectrogram - means) / stds

    mfccs = tf.signal.mfccs_from_log_mel_spectrograms(log_mel_spectrogram)[..., :13]

    return tf.expand_dims(mfccs, -1)

In [ ]:
def load_data_MFCC(file_path):
    label = get_label(file_path)
    spectrogram = process_audio_MFCC(file_path)
    return spectrogram, label

In [ ]:
train_ds = tf.data.Dataset.list_files(f"{DATASET_PATH}/train/*/*.wav")
val_ds = tf.data.Dataset.list_files(f"{DATASET_PATH}/validation/*/*.wav")
test_ds = tf.data.Dataset.list_files(f"{DATASET_PATH}/test/*/*.wav")

In [ ]:
train_ds_log_mel = (train_ds
    .map(load_data_log_mel, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

train_ds_MFCC = (train_ds
    .map(load_data_MFCC, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds_log_mel = (val_ds
    .map(load_data_log_mel, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds_MFCC = (val_ds
    .map(load_data_MFCC, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds_log_mel = (test_ds
    .map(load_data_log_mel, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds_MFCC = (test_ds
    .map(load_data_MFCC, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
for spec, label in train_ds_log_mel.take(1):
    input_shape_log_mel = spec[0].shape
    print("Input shape:", spec.shape)
    print("Label shape:", label.shape)
    plt.figure(figsize=(10, 4))
    plt.imshow(tf.transpose(spec[0], perm=[1, 0, 2])[:, :, 0], aspect='auto', origin='lower')
    plt.title(f"Log-Mel Spectrogram - Label: {CLASS_NAMES[label[0]]}")
    plt.colorbar()
    plt.show()

In [ ]:
print("Log-Mel Spectrogram input shape:", input_shape_log_mel)

In [ ]:
for spec, label in train_ds_MFCC.take(1):
    input_shape_MFCC = spec[0].shape
    print("Input shape:", spec.shape)
    print("Label shape:", label.shape)
    plt.figure(figsize=(10, 4))
    plt.imshow(tf.transpose(spec[0], perm=[1, 0, 2])[:, :, 0], aspect='auto', origin='lower')
    plt.title(f"MFCC - Label: {CLASS_NAMES[label[0]]}")
    plt.colorbar()
    plt.show()

In [ ]:
print("MFCC input shape:", input_shape_MFCC)

## Training:

### Log-Mel Spectrogram:

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)
random.seed(42)
np.random.seed(42)
os.environ['PYTHONHASHSEED'] = '42'

In [ ]:
model_log_mel = tf.keras.models.Sequential([
    # Input:
    tf.keras.layers.Input(input_shape_log_mel),

    # Bloco de Convolução 1
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Bloco de Convolução 2
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Bloco de Convolução 3
    tf.keras.layers.Conv2D(128, kernel_size=(3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Flatten:
    tf.keras.layers.Flatten(),

    # Bloco Denso para classificação:
    tf.keras.layers.Dense(256),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Dense(128),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')
])

model_log_mel.summary()

In [ ]:
model_log_mel.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_log_mel.keras', monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6)
]

In [ ]:
history_log_mel = model_log_mel.fit(train_ds_log_mel, validation_data=val_ds_log_mel, callbacks=callbacks, epochs=100, verbose=1)

### MFCC:

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)
random.seed(42)
np.random.seed(42)
os.environ['PYTHONHASHSEED'] = '42'

In [ ]:
model_MFCC = tf.keras.models.Sequential([
    # Input:
    tf.keras.layers.Input(input_shape_MFCC),

    # Bloco de Convolução 1
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Bloco de Convolução 2
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Bloco de Convolução 3
    tf.keras.layers.Conv2D(128, kernel_size=(3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Flatten:
    tf.keras.layers.Flatten(),

    # Bloco Denso para classificação:
    tf.keras.layers.Dense(128),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Dense(64),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')
])

model_MFCC.summary()

In [ ]:
model_MFCC.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_MFCC.keras', monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6)
]

In [ ]:
history_MFCC = model_MFCC.fit(train_ds_MFCC, validation_data=val_ds_MFCC, callbacks=callbacks, epochs=100, verbose=1)

## Testing:

### Log-Mel Spectrogram:

In [ ]:
acc = history_log_mel.history['accuracy']
val_acc = history_log_mel.history['val_accuracy']
loss = history_log_mel.history['loss']
val_loss = history_log_mel.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, 'o-', label='Acurácia de Treino')
plt.plot(epochs_range, val_acc, 'o-', label='Acurácia de Validação')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.grid(True)
plt.legend(loc='lower right')
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, 'o-', label='Loss de Treino')
plt.plot(epochs_range, val_loss, 'o-', label='Loss de Validação')
plt.title('Perda (Loss) de Treino e Validação')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.grid(True)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
model_log_mel = tf.keras.models.load_model('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_log_mel.keras')

y_pred_probs = model_log_mel.predict(test_ds_log_mel)
y_pred_indices = np.argmax(y_pred_probs, axis=1)

y_true_indices = np.concatenate([y for x, y in test_ds_log_mel], axis=0)

print("\n--- Relatório de Classificação ---")
report = classification_report(y_true_indices, y_pred_indices, target_names=CLASS_NAMES)
print(report)

In [ ]:
cm = confusion_matrix(y_true_indices, y_pred_indices)
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = cm.astype('float') / row_sums

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.4f',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title('Matriz de Confusão')
plt.ylabel('Label Verdadeiro')
plt.xlabel('Label Previsto')
plt.show()

### MFCC:

In [ ]:
acc = history_MFCC.history['accuracy']
val_acc = history_MFCC.history['val_accuracy']
loss = history_MFCC.history['loss']
val_loss = history_MFCC.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, 'o-', label='Acurácia de Treino')
plt.plot(epochs_range, val_acc, 'o-', label='Acurácia de Validação')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.grid(True)
plt.legend(loc='lower right')
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, 'o-', label='Loss de Treino')
plt.plot(epochs_range, val_loss, 'o-', label='Loss de Validação')
plt.title('Perda (Loss) de Treino e Validação')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.grid(True)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
model_MFCC = tf.keras.models.load_model('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_MFCC.keras')

y_pred_probs = model_MFCC.predict(test_ds_MFCC)
y_pred_indices = np.argmax(y_pred_probs, axis=1)

y_true_indices = np.concatenate([y for x, y in test_ds_MFCC], axis=0)

print("\n--- Relatório de Classificação ---")
report = classification_report(y_true_indices, y_pred_indices, target_names=CLASS_NAMES)
print(report)

In [ ]:
cm = confusion_matrix(y_true_indices, y_pred_indices)
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = cm.astype('float') / row_sums

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.4f',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title('Matriz de Confusão')
plt.ylabel('Label Verdadeiro')
plt.xlabel('Label Previsto')
plt.show()

## Gerando modelo .tflite (float16):

### Log-Mel Spectrogram:

In [ ]:
log_mel_model = tf.keras.models.load_model('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_log_mel.keras')

class EndToEndModel(tf.Module):
    def __init__(self, model):
        self.model = model
        self.sample_rate = SAMPLE_RATE
        self.n_fft = N_FFT
        self.hop_length = HOP_LENGTH
        self.n_mels = N_MELS

        self.mel_matrix = tf.signal.linear_to_mel_weight_matrix(
            num_mel_bins=self.n_mels,
            num_spectrogram_bins=self.n_fft // 2 + 1,
            sample_rate=self.sample_rate,
            lower_edge_hertz=20,
            upper_edge_hertz=self.sample_rate / 2
        )

    @tf.function(input_signature=[tf.TensorSpec(shape=[1, 16000], dtype=tf.float32)])
    def __call__(self, audio):
        waveform = audio[0, :self.sample_rate]

        stft = tf.signal.stft(waveform, frame_length=self.n_fft, frame_step=self.hop_length, fft_length=self.n_fft)
        spectrogram = tf.abs(stft)
        mel_spectrogram = tf.tensordot(spectrogram, self.mel_matrix, 1)
        log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

        means = tf.math.reduce_mean(log_mel_spectrogram)
        stds = tf.math.reduce_std(log_mel_spectrogram)

        stds = tf.where(stds < 1e-6, 1e-6, stds)

        log_mel_spectrogram = (log_mel_spectrogram - means) / stds

        model_input = tf.expand_dims(log_mel_spectrogram, axis=-1)
        model_input = tf.expand_dims(model_input, axis=0)
        return self.model(model_input)
        
end_to_end_wrapper = EndToEndModel(log_mel_model)
converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [end_to_end_wrapper.__call__.get_concrete_function()]
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_log_mel.tflite', 'wb') as f:
    f.write(tflite_model)

### MFCC:

In [ ]:
MFCC_model = tf.keras.models.load_model('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_MFCC.keras')

class EndToEndModel(tf.Module):
    def __init__(self, model):
        self.model = model
        self.sample_rate = SAMPLE_RATE
        self.n_fft = N_FFT
        self.hop_length = HOP_LENGTH
        self.n_mels = N_MELS

        self.mel_matrix = tf.signal.linear_to_mel_weight_matrix(
            num_mel_bins=self.n_mels,
            num_spectrogram_bins=self.n_fft // 2 + 1,
            sample_rate=self.sample_rate,
            lower_edge_hertz=20,
            upper_edge_hertz=self.sample_rate / 2
        )

    @tf.function(input_signature=[tf.TensorSpec(shape=[1, 16000], dtype=tf.float32)])
    def __call__(self, audio):
        waveform = audio[0, :self.sample_rate]

        stft = tf.signal.stft(waveform, frame_length=self.n_fft, frame_step=self.hop_length, fft_length=self.n_fft)
        spectrogram = tf.abs(stft)
        mel_spectrogram = tf.tensordot(spectrogram, self.mel_matrix, 1)
        log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

        means = tf.math.reduce_mean(log_mel_spectrogram)
        stds = tf.math.reduce_std(log_mel_spectrogram)

        stds = tf.where(stds < 1e-6, 1e-6, stds)

        log_mel_spectrogram = (log_mel_spectrogram - means) / stds

        mfccs = tf.signal.mfccs_from_log_mel_spectrograms(log_mel_spectrogram)[..., :13]

        model_input = tf.expand_dims(mfccs, axis=-1)
        model_input = tf.expand_dims(model_input, axis=0)
        return self.model(model_input)
        
end_to_end_wrapper = EndToEndModel(MFCC_model)
converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [end_to_end_wrapper.__call__.get_concrete_function()]
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_MFCC.tflite', 'wb') as f:
    f.write(tflite_model)

## Testing .tflite models:

### Log-Mel Spectrogram:

In [ ]:
TFLITE_MODEL_PATH = Path('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_log_mel.tflite')

print(f"Carregando modelo TFLite de: {TFLITE_MODEL_PATH}")
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_MODEL_PATH))

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print("\n--- Detalhes do Modelo TFLite ---")
print(f"Entrada (Shape): {input_details['shape']}")
print(f"Entrada (Tipo): {input_details['dtype']}")
print(f"Saída (Shape): {output_details['shape']}")
print(f"Saída (Tipo): {output_details['dtype']}")

In [ ]:
def load_wav_for_test(file_path):
    audio_binary = tf.io.read_file(file_path)
    audio, _ = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)

    audio_len = tf.shape(audio)[0]
    if audio_len > SAMPLE_RATE:
        audio = audio[:SAMPLE_RATE]
    else:
        audio = tf.pad(audio, [[0, SAMPLE_RATE - audio_len]])

    audio = tf.expand_dims(audio, axis=0)

    return audio, get_label(file_path)

test_ds_lite = test_ds.map(load_wav_for_test).batch(1)

y_true = []
y_pred = []

print("Iniciando avaliação no dataset de teste...")
for data_batch, labels_batch in test_ds_lite:
    for i in range(data_batch.shape[0]):
        sample_data = data_batch[i]
        input_tensor = sample_data.numpy().astype(input_details['dtype'])
        interpreter.set_tensor(input_details['index'], input_tensor)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details['index'])
        predicted_label_index = np.argmax(output_data)
        y_pred.append(predicted_label_index)
        y_true.append(labels_batch[i].numpy())
        
print("\nAvaliação concluída.")

In [ ]:
print("\n--- Relatório de Classificação ---\n")
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print(report)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = cm.astype('float') / row_sums

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.4f',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.title('Matriz de Confusão - Modelo TFLite Float16')
plt.ylabel('Label Verdadeiro')
plt.xlabel('Label Previsto')
plt.tight_layout()
plt.show()

### MFCC:

In [ ]:
TFLITE_MODEL_PATH = Path('../Projects/KWS-rasp-zero/KWS-RaspZero2W/Log_mel_&_MFCC/models/model_MFCC.tflite')

print(f"Carregando modelo TFLite de: {TFLITE_MODEL_PATH}")
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_MODEL_PATH))

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print("\n--- Detalhes do Modelo TFLite ---")
print(f"Entrada (Shape): {input_details['shape']}")
print(f"Entrada (Tipo): {input_details['dtype']}")
print(f"Saída (Shape): {output_details['shape']}")
print(f"Saída (Tipo): {output_details['dtype']}")